# Lecture 7: Retrieval-Augmented Generation (RAG) - Demonstration

This notebook demonstrates the key concepts from Lecture 7 on RAG:
1. Document Loading
2. Document Splitting/Chunking
3. Embeddings and Vector Stores
4. Retrieval Techniques
5. Question Answering with RAG
6. Evaluation and Best Practices

**Prerequisites**: OpenAI API key (or other LLM provider)

## Setup and Installation

Install required packages:

In [1]:
# Install required packages
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters
!pip install -q chromadb pypdf beautifulsoup4 sentence-transformers faiss-cpu
!pip install -q openai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
# Import libraries
import os
import numpy as np
from getpass import getpass

# Set OpenAI API key
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✓ Setup complete!")

✓ Setup complete!


## 1. Document Loading

Demonstrate loading documents from various sources.

### 1.1 Loading from Web (URL)

In [8]:
!pip install pdfminer.six

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 126.8 MB/s eta 0:00:00


In [9]:
import bs4
from langchain_community.document_loaders import WebBaseLoader, PDFMinerLoader

# Optional: filter to main content on web pages
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))

# List of web pages to load
web_pages = [
    "https://finance.yahoo.com/markets/",
    "https://thebudgetmom.com/the-secret-to-personal-finance-i-never-learned-about-in-business-school/",
    "https://thebudgetmom.com/the-hidden-cost-of-financial-comparison-in-the-age-of-social-media/",
    "https://thebudgetmom.com/the-truth-about-tax-refunds/",
    "https://thebudgetmom.com/why-more-money-doesnt-fix-bad-money-habits-and-what-actually-does/",
    "https://thebudgetmom.com/recovering-financially-after-the-holidays/"
]

# Load web pages
web_loader = WebBaseLoader(web_paths=web_pages)
web_docs = web_loader.load()

# Load PDF document using PDFMinerLoader from langchain_community
pdf_loader = PDFMinerLoader("https://cdn.bookey.app/files/pdf/book/en/personal-finance-for-dummies.pdf")
pdf_docs = pdf_loader.load()

# Combine all documents
all_docs = web_docs + pdf_docs

print(f"Total documents loaded: {len(all_docs)}")

# Preview first document
print(f"\nFirst document characters: {len(all_docs[0].page_content):,}")
print(f"First 500 characters:\n{all_docs[0].page_content[:500]}...")
print(f"Metadata: {all_docs[0].metadata}")

Total documents loaded: 7

First document characters: 29,996
First 500 characters:














































































































































































































































Markets: World Indexes, Futures, Bonds, Currencies, Stocks & ETFs - Yahoo Finance 



    Oops, something went wrong     Skip to navigation   Skip to main content   Skip to right column              News   Today's news   US   Politics   World   Weather   Climate...
Metadata: {'source': 'https://finance.yahoo.com/markets/', 'title': 'Markets: World Indexes, Futures, Bonds, Currencies, Stocks & ETFs - Yahoo Finance', 'description': "Yahoo Finance's market overview provides up to the minute charts, data, analysis and news about US and world markets, futures, bonds, options, currencies and more.", 'language': 'en-US'}


In [14]:
docs = all_docs

### 1.2 Loading PDF Files (Optional)

Uncomment if you have PDF files to load.

## 2. Document Splitting

Split documents into smaller chunks for embedding and retrieval.

### 2.1 RecursiveCharacterTextSplitter (Recommended)

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,           # Maximum chunk size (characters)
    chunk_overlap=200,          # Overlap between chunks
    add_start_index=True,       # Track position in original document
    separators=["\n\n", "\n", " ", ""]  # Try separators in order
)

# Split the loaded documents
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} chunks")
print(f"\nChunk 0 length: {len(all_splits[0].page_content)} characters")
print(f"Chunk 0 metadata: {all_splits[0].metadata}")
print(f"\nFirst chunk content:\n{all_splits[0].page_content}")

Split blog post into 328 chunks

Chunk 0 length: 81 characters
Chunk 0 metadata: {'source': 'https://finance.yahoo.com/markets/', 'title': 'Markets: World Indexes, Futures, Bonds, Currencies, Stocks & ETFs - Yahoo Finance', 'description': "Yahoo Finance's market overview provides up to the minute charts, data, analysis and news about US and world markets, futures, bonds, options, currencies and more.", 'language': 'en-US', 'start_index': 238}

First chunk content:
Markets: World Indexes, Futures, Bonds, Currencies, Stocks & ETFs - Yahoo Finance


### 2.2 Visualizing Chunk Overlap

In [11]:
# Demonstrate overlap with simple example
simple_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

sample_text = """An emergency fund acts as a financial safety net for unexpected expenses like medical bills or job loss.
Most financial advisors recommend saving three to six months of living expenses in a liquid, accessible account.
Starting small with even $500 can protect you from going into debt when life throws you a curveball."""

simple_chunks = simple_splitter.split_text(sample_text)

print(f"Created {len(simple_chunks)} chunks with overlap:\n")
for i, chunk in enumerate(simple_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}")
    print(f"{'-'*80}")

Created 9 chunks with overlap:

Chunk 1 (48 chars): An emergency fund acts as a financial safety net
--------------------------------------------------------------------------------
Chunk 2 (49 chars): net for unexpected expenses like medical bills or
--------------------------------------------------------------------------------
Chunk 3 (18 chars): bills or job loss.
--------------------------------------------------------------------------------
Chunk 4 (49 chars): Most financial advisors recommend saving three to
--------------------------------------------------------------------------------
Chunk 5 (43 chars): three to six months of living expenses in a
--------------------------------------------------------------------------------
Chunk 6 (32 chars): in a liquid, accessible account.
--------------------------------------------------------------------------------
Chunk 7 (45 chars): Starting small with even $500 can protect you
---------------------------------------------------

### 2.3 Comparing Different Splitters

In [16]:
from langchain_text_splitters import CharacterTextSplitter

# CharacterTextSplitter (splits only on specified separator)
char_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator="\n\n"  # Only split on double newlines
)

char_splits = char_splitter.split_documents(docs)

print(f"RecursiveCharacterTextSplitter: {len(all_splits)} chunks")
print(f"CharacterTextSplitter: {len(char_splits)} chunks")
print(f"\nRecursive splitter creates more uniform chunks by trying multiple separators.")

RecursiveCharacterTextSplitter: 328 chunks
CharacterTextSplitter: 244 chunks

Recursive splitter creates more uniform chunks by trying multiple separators.


## 3. Embeddings and Vector Stores

Create embeddings and store them in a vector database.

### 3.1 Computing Embeddings

In [17]:
from langchain_openai import OpenAIEmbeddings

# Initialize embedding model
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Embed three sample sentences
sentence1 = "Pay yourself first by setting aside savings before spending on anything else"
sentence2 = "Automating contributions to savings removes the temptation to skip a month"
sentence3 = "Even small weekly deposits compound into significant wealth over time"

embedding1 = embedding_model.embed_query(sentence1)
embedding2 = embedding_model.embed_query(sentence2)
embedding3 = embedding_model.embed_query(sentence3)

print(f"Embedding dimension: {len(embedding1)}")
print(f"\nFirst 10 values of embedding1: {embedding1[:10]}")

Embedding dimension: 1536

First 10 values of embedding1: [0.007771211676299572, -0.04433412849903107, 0.01995035633444786, 0.01798844523727894, -0.007962306961417198, -0.00014869608276057988, 0.0090515511110425, 0.06283216178417206, -0.00045345339458435774, -0.03169635310769081]


### 3.2 Semantic Similarity

In [18]:
# Compute cosine similarity (dot product of normalized vectors)
# OpenAI embeddings are already normalized

similarity_1_2 = np.dot(embedding1, embedding2)
similarity_1_3 = np.dot(embedding1, embedding3)
similarity_2_3 = np.dot(embedding2, embedding3)

print("Semantic Similarity Scores (cosine similarity):")
print(f"'{sentence1}' vs '{sentence2}': {similarity_1_2:.4f} (synonyms - HIGH)")
print(f"'{sentence1}' vs '{sentence3}': {similarity_1_3:.4f} (unrelated - LOW)")
print(f"'{sentence2}' vs '{sentence3}': {similarity_2_3:.4f} (unrelated - LOW)")

Semantic Similarity Scores (cosine similarity):
'Pay yourself first by setting aside savings before spending on anything else' vs 'Automating contributions to savings removes the temptation to skip a month': 0.5340 (synonyms - HIGH)
'Pay yourself first by setting aside savings before spending on anything else' vs 'Even small weekly deposits compound into significant wealth over time': 0.3808 (unrelated - LOW)
'Automating contributions to savings removes the temptation to skip a month' vs 'Even small weekly deposits compound into significant wealth over time': 0.4340 (unrelated - LOW)


### 3.3 Creating a Vector Store with Chroma

In [19]:
from langchain_community.vectorstores import Chroma

# Create vector store from document chunks
persist_directory = './chroma_db'

vectordb = Chroma.from_documents(
    documents=all_splits,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print(f"✓ Vector store created with {vectordb._collection.count()} document chunks")
print(f"✓ Persisted to: {persist_directory}")

✓ Vector store created with 328 document chunks
✓ Persisted to: ./chroma_db


### 3.4 Loading Existing Vector Store

In [20]:
# Load existing vector store (no need to re-embed)
vectordb_loaded = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

print(f"✓ Loaded vector store with {vectordb_loaded._collection.count()} documents")

✓ Loaded vector store with 328 documents


/tmp/ipykernel_336/1684523185.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb_loaded = Chroma(


## 4. Retrieval Techniques

Demonstrate different retrieval methods.

### 4.1 Basic Similarity Search

In [21]:
question = "Capy, I just got a $500 bonus at work — should I put it toward my emergency fund or pay off my credit card debt first?"

# Retrieve top-k similar documents
docs_retrieved = vectordb.similarity_search(question, k=3)

print(f"Query: {question}")
print(f"\nRetrieved {len(docs_retrieved)} documents:\n")

for i, doc in enumerate(docs_retrieved):
    print(f"{'='*80}")
    print(f"Document {i+1}")
    print(f"{'='*80}")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Start Index: {doc.metadata.get('start_index', 'N/A')}")
    print(f"\nContent:\n{doc.page_content[:300]}...\n")

Query: Capy, I just got a $500 bonus at work — should I put it toward my emergency fund or pay off my credit card debt first?

Retrieved 3 documents:

Document 1
Source: https://thebudgetmom.com/recovering-financially-after-the-holidays/
Start Index: 6773

Content:
Can you pay more than the minimum right now?
Or is maintaining payments while rebuilding cash flow the smarter move?

Debt recovery doesn’t have to be aggressive to be effective. It has to be consistent.
Step Seven: Rebuild Savings Slowly
If you dipped into savings or emptied a sinking fund during t...

Document 2
Source: https://cdn.bookey.app/files/pdf/book/en/personal-finance-for-dummies.pdf
Start Index: 36468

Content:
Paying Off High-Interest Debt

- Paying off high-interest debts (like credit card debt) can
yield guaranteed returns equal to the interest rate on the debt.
- This approach is particularly effective compared to
investing elsewhere, noting that consumer debt interest is not
tax-deductible.

Taking Ad...

Do

### 4.2 Similarity Search with Scores

In [22]:
# Get documents with similarity scores
docs_with_scores = vectordb.similarity_search_with_score(question, k=3)

print(f"Query: {question}\n")

for i, (doc, score) in enumerate(docs_with_scores):
    print(f"Document {i+1} - Similarity Score: {score:.4f}")
    print(f"Content preview: {doc.page_content[:200]}...")
    print(f"{'-'*80}\n")

Query: Capy, I just got a $500 bonus at work — should I put it toward my emergency fund or pay off my credit card debt first?

Document 1 - Similarity Score: 1.0870
Content preview: Can you pay more than the minimum right now?
Or is maintaining payments while rebuilding cash flow the smarter move?

Debt recovery doesn’t have to be aggressive to be effective. It has to be consiste...
--------------------------------------------------------------------------------

Document 2 - Similarity Score: 1.1102
Content preview: Paying Off High-Interest Debt

- Paying off high-interest debts (like credit card debt) can
yield guaranteed returns equal to the interest rate on the debt.
- This approach is particularly effective c...
--------------------------------------------------------------------------------

Document 3 - Similarity Score: 1.1205
Content preview: Smaller debt payments temporarily
Simplified budgets
Fewer categories
Less tracking
More margin

This season is about consistency, not i

### 4.3 Maximum Marginal Relevance (MMR)

Balance relevance and diversity in retrieved results.

In [23]:
question_mmr = "I'm 25 with $200/month to invest — should I prioritize maxing out my Roth IRA or put money into a brokerage account?"

print(f"Query: {question_mmr}\n")
print("="*80)
print("STANDARD SIMILARITY SEARCH")
print("="*80)

docs_ss = vectordb.similarity_search(question_mmr, k=3)
for i, doc in enumerate(docs_ss):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n" + "="*80)
print("MMR SEARCH (Diverse Results)")
print("="*80)

docs_mmr = vectordb.max_marginal_relevance_search(
    question_mmr,
    k=3,
    fetch_k=20  # Fetch 20 candidates, return 3 diverse ones
)

for i, doc in enumerate(docs_mmr):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}...")

print("\n💡 Note: MMR results should be more diverse than similarity search.")

Query: I'm 25 with $200/month to invest — should I prioritize maxing out my Roth IRA or put money into a brokerage account?

STANDARD SIMILARITY SEARCH

Doc 1: investments?
Answer:Diversifying retirement investments is crucial as it

helps manage risk. Different asset classes (stocks, bonds,

etc.) perform di...

Doc 2: Individual Retirement Accounts (IRAs)

Anyone earning income can contribute to an IRA. There are
annual contribution limits, which vary based on the a...

Doc 3: Recommended Money Market Mutual Funds

- Depending on your tax situation, choose money market
funds that suit your bracket (taxable, Treasury, state-s...

MMR SEARCH (Diverse Results)

Doc 1: investments?
Answer:Diversifying retirement investments is crucial as it

helps manage risk. Different asset classes (stocks, bonds,

etc.) perform di...

Doc 2: Recommended Money Market Mutual Funds

- Depending on your tax situation, choose money market
funds that suit your bracket (taxable, Treasury, state-s...

Doc 3: 

### 4.4 Metadata Filtering

Filter results based on metadata.

In [24]:
# First, let's see what metadata is available
sample_doc = all_splits[0]
print("Available metadata fields:")
for key, value in sample_doc.metadata.items():
    print(f"  {key}: {value}")

# Example: Filter by source (if you have multiple sources)
# Note: This example uses the web source we loaded
source_filter = sample_doc.metadata.get('source')

print(f"\nFiltering by source: {source_filter}")

docs_filtered = vectordb.similarity_search(
    "How do I know when to rebalance my portfolio, and what percentage should I keep in stocks vs. bonds at age 30?",
    k=3,
    filter={"source": source_filter}
)

print(f"\nRetrieved {len(docs_filtered)} filtered documents")
for i, doc in enumerate(docs_filtered):
    print(f"Doc {i+1} source: {doc.metadata.get('source')}")

Available metadata fields:
  source: https://finance.yahoo.com/markets/
  title: Markets: World Indexes, Futures, Bonds, Currencies, Stocks & ETFs - Yahoo Finance
  description: Yahoo Finance's market overview provides up to the minute charts, data, analysis and news about US and world markets, futures, bonds, options, currencies and more.
  language: en-US
  start_index: 238

Filtering by source: https://finance.yahoo.com/markets/

Retrieved 3 filtered documents
Doc 1 source: https://finance.yahoo.com/markets/
Doc 2 source: https://finance.yahoo.com/markets/
Doc 3 source: https://finance.yahoo.com/markets/


## 5. Question Answering with RAG

Combine retrieval with LLM generation.

### 5.1 Simple RAG with LangChain Agent

In [25]:
# No need for langchain_classic - we'll use the modern API
# Just ensure we have the latest langchain packages
!pip install -q --upgrade langchain langchain-openai langchain-community

In [26]:
from langchain.agents import create_agent
from langchain.tools import tool

# Create retrieval tool
@tool
def retrieve_context(query: str) -> str:
    """Retrieve information from the knowledge base to help answer questions."""
    print(f"Retrieving context for query: {query}")
    retrieved_docs = vectordb.similarity_search(query, k=3)
    serialized = "\n\n".join(
        f"Source: {doc.metadata}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized

# Create agent with tools
agent = create_agent(
    model="gpt-4o-mini",
    tools=[retrieve_context],
    #temperature=0,
    system_prompt="You are a helpful AI assistant. Use the retrieve_context tool to find information from the knowledge base when needed to answer questions accurately."
)

print("✓ RAG Agent created successfully!")

✓ RAG Agent created successfully!


In [27]:
# Ask a question using the agent
question = "What's the difference between index funds and ETFs, and which one makes more sense for a beginner investor?"

print(f"Question: {question}\n")
print("="*80)

# Invoke the agent
result = agent.invoke({
    "messages": [{"role": "user", "content": question}]
})

print("\n" + "="*80)
print("FINAL ANSWER:")
print("="*80)
# The final message contains the agent's response
print(result["messages"][-1].content)

Question: What's the difference between index funds and ETFs, and which one makes more sense for a beginner investor?

Retrieving context for query: difference between index funds and ETFs
Retrieving context for query: which is better for beginner investors index funds or ETFs

FINAL ANSWER:
### Difference Between Index Funds and ETFs

1. **Structure and Trading**:
   - **Index Funds**: These are mutual funds designed to track a specific index (like the S&P 500). They are bought and sold at the end of the trading day at a price that reflects the net asset value (NAV).
   - **ETFs (Exchange-Traded Funds)**: ETFs also track an index and can be traded on stock exchanges throughout the trading day, similar to individual stocks. This means their price fluctuates throughout the day based on market demand.

2. **Fees**:
   - Typically, both index funds and ETFs have lower fees compared to actively managed funds, but ETFs usually have lower expense ratios and no minimum investment requirement,

### 5.1b Streaming Agent Responses (Optional)

See intermediate steps as the agent works.

In [28]:
# Stream the agent's reasoning and responses
question = "What's dollar-cost averaging and is it a better strategy than trying to time the market?"

print(f"Question: {question}\n")
print("="*80)
print("STREAMING AGENT EXECUTION:")
print("="*80 + "\n")

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values"
):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]

    # Print different message types
    if hasattr(latest_message, 'content') and latest_message.content:
        if latest_message.type == "ai":
            print(f"\n🤖 Agent: {latest_message.content}")
    elif hasattr(latest_message, 'tool_calls') and latest_message.tool_calls:
        print(f"\n🔧 Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

print("\n" + "="*80)

Question: What's dollar-cost averaging and is it a better strategy than trying to time the market?

STREAMING AGENT EXECUTION:


🔧 Calling tools: ['retrieve_context', 'retrieve_context']
Retrieving context for query: What is dollar-cost averaging?
Retrieving context for query: Is dollar-cost averaging better than timing the market?

🤖 Agent: **Dollar-Cost Averaging (DCA)** is an investment strategy where an investor regularly invests a fixed amount of money into a particular investment, such as stocks or mutual funds, at predetermined intervals. This approach helps to mitigate the impact of market volatility because it means that the investor buys more shares when prices are low and fewer shares when prices are high, thereby potentially lowering the average cost per share over time. This strategy is particularly effective for long-term investors as it encourages a disciplined investment habit and reduces the emotional stress associated with trying to time the market. 

According to "Pe

### 5.2 RAG with Custom Prompt

In [29]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Initialize the LLM for the RAG chain
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Custom prompt template
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum. Keep the answer as concise as possible.
Always say "Thanks for asking!" at the end of the answer.

Context: {context}

Question: {question}

Helpful Answer:"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

# Create RAG chain using LCEL (LangChain Expression Language)
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = (
    {
        "context": lambda x: format_docs(vectordb.similarity_search(x["question"], k=3)),
        "question": lambda x: x["question"]
    }
    | QA_CHAIN_PROMPT
    | llm
    | StrOutputParser()
)

# Helper function for RAG
def rag_qa(question):
    # Retrieve documents
    docs = vectordb.similarity_search(question, k=3)

    # Generate answer using the chain
    answer = rag_chain.invoke({"question": question})

    return answer, docs

# Test the custom RAG
question = "I'm worried about a market crash — should I move my investments to cash, or just ride it out?"
answer, sources = rag_qa(question)

print(f"Question: {question}\n")
print(f"Answer: {answer}\n")
print(f"\nSources ({len(sources)} documents):")
for i, doc in enumerate(sources):
    print(f"  {i+1}. {doc.metadata.get('source', 'Unknown')}")

Question: I'm worried about a market crash — should I move my investments to cash, or just ride it out?

Answer: It's generally better to avoid panic selling and consider riding it out, as markets can recover over time. However, assess your individual financial situation and risk tolerance before making any decisions. Thanks for asking!


Sources (3 documents):
  1. https://cdn.bookey.app/files/pdf/book/en/personal-finance-for-dummies.pdf
  2. https://cdn.bookey.app/files/pdf/book/en/personal-finance-for-dummies.pdf
  3. https://cdn.bookey.app/files/pdf/book/en/personal-finance-for-dummies.pdf


### 5.3 Interactive RAG Q&A

In [30]:
# Function for interactive Q&A with source display
def interactive_rag(question, show_sources=True, k=3):
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")

    # Retrieve
    docs = vectordb.similarity_search(question, k=k)

    # Generate using the RAG chain
    answer = rag_chain.invoke({"question": question})

    print(f"Answer: {answer}\n")

    if show_sources:
        print(f"{'='*80}")
        print(f"Retrieved Sources ({len(docs)} documents):")
        print(f"{'='*80}")
        for i, doc in enumerate(docs):
            print(f"\n[{i+1}] {doc.page_content[:200]}...")
            print(f"    Source: {doc.metadata.get('source', 'Unknown')}")

"""
# Try multiple questions
questions = [
    "What is task decomposition?",
    "What are the challenges in long-term planning for AI agents?",
    "How does ReAct work?"
]
"""

# Try multiple questions
questions = [
    "How much of my paycheck should I be investing each month if I want to retire by 55?",
    "What's dollar-cost averaging and is it a better strategy than trying to time the market?",
    "Should I sell my individual stocks and move everything into index funds, or keep a mix of both?",
    "How do dividends work, and should I reinvest them or take them as cash?",
    "What's the difference between a traditional 401(k) and a Roth 401(k), and which one should I choose?",
    "I'm worried about a market crash — should I move my investments to cash, or just ride it out?"
]

for q in questions:
    interactive_rag(q, show_sources=True)


Question: How much of my paycheck should I be investing each month if I want to retire by 55?

Answer: To determine how much of your paycheck to invest each month for retirement by age 55, aim to save at least 15-20% of your income, adjusting based on your specific retirement goals and current savings. It's also essential to calculate the total amount needed, typically around 70-80% of your pre-retirement income. Consulting a financial advisor can provide personalized guidance based on your situation. 

Thanks for asking!

Retrieved Sources (3 documents):

[1] Preparing for Retirement

To retire comfortably, individuals need to increase their
savings rate and calculate the amount needed, typically
around 70-80% of pre-retirement income.

Utilizing Social Se...
    Source: https://cdn.bookey.app/files/pdf/book/en/personal-finance-for-dummies.pdf

[2] balanced approach to achieving financial goals.

7.Question

How can I better prepare for retirement?
Answer:Preparing for retirement inv

## 6. Evaluation and Best Practices

### 6.1 Retrieval Quality Metrics

In [31]:
# Create a simple test set
# In practice, you'd have ground truth labels

def evaluate_retrieval(query, k=5):
    """
    Evaluate retrieval quality.
    Note: This is a simplified version. In practice, you'd have ground truth.
    """
    # Retrieve documents
    docs = vectordb.similarity_search_with_score(query, k=k)

    print(f"Query: {query}")
    print(f"Retrieved {len(docs)} documents:\n")

    for i, (doc, score) in enumerate(docs):
        print(f"Rank {i+1} - Score: {score:.4f}")
        print(f"Content: {doc.page_content[:100]}...")
        print(f"{'-'*60}\n")

    # Calculate score statistics
    scores = [score for _, score in docs]
    print(f"\nScore Statistics:")
    print(f"  Mean: {np.mean(scores):.4f}")
    print(f"  Std:  {np.std(scores):.4f}")
    print(f"  Min:  {np.min(scores):.4f}")
    print(f"  Max:  {np.max(scores):.4f}")

# Test retrieval quality
evaluate_retrieval("What's the difference between a traditional 401(k) and a Roth 401(k), and which one should I choose?")

Query: What's the difference between a traditional 401(k) and a Roth 401(k), and which one should I choose?
Retrieved 5 documents:

Rank 1 - Score: 0.8894
Content: contrast, traditional IRAs allow for pre-tax contributions,

with taxes owed upon withdrawal.

8.Que...
------------------------------------------------------------

Rank 2 - Score: 0.9574
Content: investments?
Answer:Diversifying retirement investments is crucial as it

helps manage risk. Differe...
------------------------------------------------------------

Rank 3 - Score: 1.0393
Content: Types of Retirement Accounts

- 
Employer-Sponsored Plans
: These accounts, such as 401(k) plans, ar...
------------------------------------------------------------

Rank 4 - Score: 1.0888
Content: 4.Funds of funds help focus fund investors on the important

big-picture issue of asset allocation—h...
------------------------------------------------------------

Rank 5 - Score: 1.1038
Content: Chapter 11: Investing in Retirement Accounts

### 6.2 Comparing Retrieval Strategies

In [32]:
def compare_retrieval_methods(query, k=3):
    """
    Compare different retrieval strategies.
    """
    print(f"Query: {query}\n")

    # Method 1: Similarity Search
    print("="*80)
    print("METHOD 1: Similarity Search")
    print("="*80)
    docs_sim = vectordb.similarity_search(query, k=k)
    for i, doc in enumerate(docs_sim):
        print(f"\n[{i+1}] {doc.page_content[:150]}...")

    # Method 2: MMR
    print("\n" + "="*80)
    print("METHOD 2: Maximum Marginal Relevance (MMR)")
    print("="*80)
    docs_mmr = vectordb.max_marginal_relevance_search(query, k=k, fetch_k=20)
    for i, doc in enumerate(docs_mmr):
        print(f"\n[{i+1}] {doc.page_content[:150]}...")

    # Check overlap
    overlap = sum(1 for d1 in docs_sim if any(
        d1.page_content == d2.page_content for d2 in docs_mmr
    ))

    print(f"\n{'='*80}")
    print(f"Overlap: {overlap}/{k} documents are the same")
    print(f"MMR provides {'more' if overlap < k else 'similar'} diversity")

compare_retrieval_methods("Should I sell my individual stocks and move everything into index funds, or keep a mix of both?")

Query: Should I sell my individual stocks and move everything into index funds, or keep a mix of both?

METHOD 1: Similarity Search

[1] - Stocks represent ownership in companies and can offer high
potential returns.
- Over the long term, the U.S. stock market historically
returns abou...

[2] Exploring Various Fund Types

Contrary to popular belief, not all mutual funds invest solely
in stocks. Various fund types include:
- 
Money Market F...

[3] alternatives can enhance your net income.

4.Question

What are some strategies to bolster my emergency

reserves?
Answer:Maintaining accessibility an...

METHOD 2: Maximum Marginal Relevance (MMR)

[1] - Stocks represent ownership in companies and can offer high
potential returns.
- Over the long term, the U.S. stock market historically
returns abou...

[2] Selecting the Best Mutual Funds
Effective selection involves:
- Reading fund prospectuses and annual reports to understand
objectives, costs, and pas...

[3] What are annuities, and 

### 6.3 Answer Quality Assessment

In [33]:
# Simple answer quality check using LLM
from langchain_core.prompts import PromptTemplate

eval_template = """Evaluate the following answer based on the provided context.

Context: {context}

Question: {question}

Answer: {answer}

Evaluation Criteria:
1. Faithfulness: Is the answer supported by the context? (Yes/No)
2. Relevance: Does the answer address the question? (Yes/No)
3. Completeness: Is the answer complete? (Yes/No)

Provide your evaluation:"""

eval_prompt = PromptTemplate.from_template(eval_template)
eval_chain = eval_prompt | llm | StrOutputParser()

def evaluate_answer(question):
    # Generate answer
    answer, docs = rag_qa(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    print(f"Question: {question}")
    print(f"\nAnswer: {answer}")

    # Evaluate
    evaluation = eval_chain.invoke({
        "context": context,
        "question": question,
        "answer": answer
    })

    print(f"\n{'='*80}")
    print("EVALUATION:")
    print(f"{'='*80}")
    print(evaluation)

evaluate_answer("I'm worried about a market crash — should I move my investments to cash, or just ride it out?")

Question: I'm worried about a market crash — should I move my investments to cash, or just ride it out?

Answer: It's generally better to ride it out rather than move everything to cash, as market downturns can be temporary. Consider your long-term goals and risk tolerance before making any drastic changes. Thanks for asking!

EVALUATION:
1. Faithfulness: Yes  
The answer aligns with the context provided, which emphasizes the importance of not panicking during market downturns and considering long-term goals and risk tolerance.

2. Relevance: Yes  
The answer directly addresses the question about whether to move investments to cash or ride out a market crash, providing a clear recommendation to ride it out.

3. Completeness: Yes  
The answer is complete as it not only suggests riding out the downturn but also encourages consideration of long-term goals and risk tolerance, which are important factors in investment decisions.

Overall Evaluation: The answer is faithful, relevant, and com

### 6.4 Chunk Size Experimentation

In [34]:
# Test different chunk sizes
chunk_sizes = [500, 1000, 1500]

print("Testing different chunk sizes:\n")

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.2),  # 20% overlap
        add_start_index=True
    )

    splits = splitter.split_documents(docs)

    print(f"Chunk size: {size}")
    print(f"  Total chunks: {len(splits)}")
    print(f"  Avg chunk length: {np.mean([len(s.page_content) for s in splits]):.1f}")
    print(f"  Min chunk length: {min([len(s.page_content) for s in splits])}")
    print(f"  Max chunk length: {max([len(s.page_content) for s in splits])}")
    print()

print("💡 Smaller chunks → more granular retrieval but may lose context")
print("💡 Larger chunks → more context but less precise retrieval")

Testing different chunk sizes:

Chunk size: 500
  Total chunks: 670
  Avg chunk length: 420.4
  Min chunk length: 27
  Max chunk length: 499

Chunk size: 1000
  Total chunks: 328
  Avg chunk length: 886.1
  Min chunk length: 34
  Max chunk length: 998

Chunk size: 1500
  Total chunks: 221
  Avg chunk length: 1342.6
  Min chunk length: 81
  Max chunk length: 1499

💡 Smaller chunks → more granular retrieval but may lose context
💡 Larger chunks → more context but less precise retrieval


## 7. Advanced: Alternative Embedding Models

### 7.1 Using HuggingFace Embeddings (Free)

In [35]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Use a free open-source embedding model
hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Test embeddings
test_text = "How do I build a diversified investment portfolio on a beginner's budget?"
hf_embedding = hf_embeddings.embed_query(test_text)

print(f"HuggingFace Embedding dimension: {len(hf_embedding)}")
print(f"First 10 values: {hf_embedding[:10]}")

# Create vector store with HuggingFace embeddings
vectordb_hf = Chroma.from_documents(
    documents=all_splits[:50],  # Use subset for faster demo
    embedding=hf_embeddings,
    persist_directory='./chroma_db_hf'
)

print(f"\n✓ Created vector store with HuggingFace embeddings")
print(f"✓ Contains {vectordb_hf._collection.count()} documents")

/tmp/ipykernel_336/2211414050.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace Embedding dimension: 384
First 10 values: [0.06767924875020981, -0.003672058694064617, -0.11593425273895264, 0.0014676066348329186, -0.031121421605348587, -0.036642469465732574, 0.05849310755729675, 0.012906752526760101, -0.08007699996232986, 0.0994512215256691]

✓ Created vector store with HuggingFace embeddings
✓ Contains 50 documents


## 8. Summary and Key Takeaways

### What we demonstrated:

1. **Document Loading**: Loaded web content using LangChain's WebBaseLoader
2. **Document Splitting**: Used RecursiveCharacterTextSplitter with customizable chunk size and overlap
3. **Embeddings**: Created semantic embeddings using OpenAI's text-embedding-3-small
4. **Vector Stores**: Stored and indexed embeddings in Chroma for efficient similarity search
5. **Retrieval**: Demonstrated multiple retrieval strategies:
   - Basic similarity search
   - Maximum Marginal Relevance (MMR) for diversity
   - Metadata filtering
6. **Question Answering**: Built RAG pipelines with custom prompts
7. **Evaluation**: Assessed retrieval and answer quality

### Best Practices:

- **Chunk Size**: Start with 1000 characters, 20% overlap
- **Retrieval**: Use k=3-5 documents for most queries
- **Prompts**: Always instruct LLM to use context and avoid hallucination
- **Evaluation**: Test with real questions from your domain
- **Iteration**: Experiment with chunk sizes, k values, and prompts

### Next Steps:

- Try with your own documents
- Experiment with different embedding models
- Implement hybrid search (keyword + semantic)
- Add reranking for better precision
- Build a chat interface with conversation history

## 9. Cleanup (Optional)

Remove created files if needed.

In [36]:
# Uncomment to clean up vector databases
# import shutil
# import os

# if os.path.exists('./chroma_db'):
#     shutil.rmtree('./chroma_db')
#     print("✓ Removed ./chroma_db")

# if os.path.exists('./chroma_db_hf'):
#     shutil.rmtree('./chroma_db_hf')
#     print("✓ Removed ./chroma_db_hf")